# Streamlit Web应用部署指南

> **对应模块**: `src/app.py` (Web界面)
> **前置条件**: `pip install streamlit python-dotenv`

---

## 什么是Streamlit？

Streamlit是一个Python Web框架，专为数据科学和ML项目设计。
- **零前端知识**: 不需要HTML/CSS/JavaScript
- **热重载**: 代码修改后自动刷新
- **丰富组件**: 上传文件、滑块、聊天框、图表，一行代码搞定

## 启动智析Web应用

In [ ]:
# 在终端中运行以下命令 (不是在Notebook中):
# cd zhixi
# streamlit run src/app.py

# 如果是在Notebook中测试，可以用以下方式:
print("=" * 60)
print("智析 Web应用启动指南")
print("=" * 60)

print("""
步骤1: 安装依赖
  cd zhixi
  pip install -r requirements.txt

步骤2: 配置环境变量
  复制 .env.example 为 .env
  填入你的 OpenAI API Key (或配置Ollama)

步骤3: 启动应用
  streamlit run src/app.py

步骤4: 打开浏览器
  默认地址: http://localhost:8501

步骤5: 使用流程
  1. 在侧边栏选择LLM模式 (OpenAI/Ollama)
  2. 在"文档解析"标签页上传PDF
  3. 在"NLP分析"标签页执行分析
  4. 在"知识图谱"标签页构建图谱
  5. 在"智能问答"标签页提问
""")

print("应用截图说明:")
print("  - 顶部: 项目标题 '📊 智析'")
print("  - 侧边栏: LLM配置 + RAG参数")
print("  - 主区域: 4个标签页 (文档解析/NLP分析/知识图谱/智能问答)")

## app.py 代码结构解析

### 核心函数:
| 函数 | 功能 |
|------|------|
| `init_session_state()` | 初始化Streamlit会话状态 |
| `render_sidebar()` | 渲染侧边栏 (LLM配置+RAG参数) |
| `render_header()` | 渲染页面标题 |
| `render_upload_section()` | 文档上传和解析 |
| `render_nlp_section()` | NLP分析 |
| `render_kg_section()` | 知识图谱构建 |
| `render_rag_section()` | RAG问答 |
| `_parse_document()` | 调用doc_parser解析PDF |
| `_init_rag()` | 初始化RAGEngine并导入文档 |
| `_ask_question()` | 调用RAGEngine回答问题 |

### 关键Streamlit组件:
```python
st.file_uploader()     # 文件上传
st.chat_input()        # 聊天输入框
st.chat_message()      # 聊天气泡
st.tabs()              # 标签页
st.spinner()           # 加载动画
st.metric()            # 指标卡片
st.expander()          # 可折叠区域
st.session_state       # 会话状态管理
```

In [ ]:
# ========================================
# 模拟Web应用的数据流
# ========================================

import sys, os
sys.path.insert(0, '..')
os.makedirs('../data/processed', exist_ok=True)

print("模拟Web应用的完整数据流:\n")

# Step 1: 用户上传PDF → doc_parser
print("[Tab 1: 文档解析]")
from src.doc_parser import DocumentParser

test_pdf = '../data/sample_docs/test.pdf'
if os.path.exists(test_pdf):
    parser = DocumentParser(test_pdf, output_dir='../data/processed')
    result = parser.parse()
    chunks = parser.get_text_chunks(chunk_size=500, chunk_overlap=50)
    print(f"  页数: {result.total_pages}, 字符: {len(result.full_text)}, 块数: {len(chunks)}")
else:
    print("  请先运行 02_ocr_experiment.ipynb 创建测试PDF")
    chunks = []

# Step 2: NLP分析 → nlp_pipeline  
print("\n[Tab 2: NLP分析]")
print("  (需要安装transformers/keybert，首次运行需下载模型)")
print("  功能: 关键词提取、实体识别、摘要生成、词云")

# Step 3: 知识图谱 → knowledge_graph
print("\n[Tab 3: 知识图谱]")
from src.knowledge_graph import KnowledgeGraphBuilder
kg = KnowledgeGraphBuilder()
kg.add_entities([("智析", "PRODUCT"), ("RAG", "TECH"), ("ChromaDB", "TECH")])
kg.add_relation("智析", "uses", "RAG")
kg.add_relation("智析", "uses", "ChromaDB")
stats = kg.get_stats()
print(f"  节点: {stats.node_count}, 关系: {stats.edge_count}")

# Step 4: RAG问答 → rag_engine
print("\n[Tab 4: RAG问答]")
print("  流程: 初始化引擎 → 导入文档块 → 用户提问 → 检索+生成")
if chunks:
    print(f"  已准备好 {len(chunks)} 个文档块，等待初始化RAG引擎...")

print("\n" + "="*60)
print("Web应用数据流模拟完成!")
print("运行 'streamlit run src/app.py' 启动真实应用")

---
## 部署到互联网 (可选)

### 方案1: Streamlit Community Cloud (免费)
1. 将项目推送到GitHub
2. 访问 share.streamlit.io
3. 连接GitHub仓库，选择 `src/app.py`
4. 自动生成公开链接

### 方案2: HuggingFace Spaces (免费)
1. 注册 huggingface.co 账号
2. 创建新Space，选择Streamlit框架
3. 上传项目文件
4. 自动生成公开链接

### 简历展示技巧
- 提供**在线Demo链接** (面试官不用安装任何东西)
- GitHub README中放**截图和GIF**
- 准备**2分钟演示视频**
- 准备**30秒电梯演讲**: 
  > "我做了一个多模态文档智能分析平台，能解析论文/研报的文字和图表，
  > 构建知识图谱，通过RAG增强的LLM对话回答深度问题。"

---

## 项目总结: 你学到了什么

### 技术栈全景:
```
Python基础 → NumPy/Pandas → Matplotlib → Scikit-learn
                                              ↓
                                    PyMuPDF/pdfplumber (CV)
                                              ↓
                                    HuggingFace/KeyBERT (NLP)
                                              ↓
                                    NetworkX (知识图谱)
                                              ↓
                                    LangChain/ChromaDB (RAG)
                                              ↓
                                    Streamlit (Web应用)
```

### 四大AI领域的融合:
- **CV**: PDF解析、OCR、图表识别
- **NLP**: 实体识别、关键词、摘要
- **数据挖掘**: 知识图谱、主题聚类
- **LLM应用**: RAG问答、Prompt工程

恭喜你完成了整个学习项目！